### Imports and train - test split

In [1]:
import pandas as pd
import joblib 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier


In [2]:

data = pd.read_csv('../data/datasets/processed_dataset_v3.csv')
X = data.drop(['label'],axis=1)
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 80-20 split
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, '../models/scaler.pkl')

['../models/scaler.pkl']

#### Logistic Regression

Training and fitting/ pickel dump/ accuracy score

In [3]:
lr = LogisticRegression(max_iter=500)
lr.fit(X_train_scaled, y_train)
joblib.dump(lr,'../models/logistic.pkl')

lr_pred = lr.predict(X_test_scaled)
lr_acc = accuracy_score(y_test, lr_pred)
print(f"LogisticRegression Accuracy: {lr_acc * 100:.2f}%")
print("\nLogisticRegression Classification Report:\n", classification_report(y_test, lr_pred))

LogisticRegression Accuracy: 82.49%

LogisticRegression Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.86      0.81      8114
           1       0.89      0.80      0.84     11070

    accuracy                           0.82     19184
   macro avg       0.82      0.83      0.82     19184
weighted avg       0.83      0.82      0.83     19184



Confusion Matrix and heatmap


In [4]:
lrcm = confusion_matrix(y_test, lr_pred)
plt.figure(figsize=(6,4))
sns.heatmap(lrcm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix for LogisticRegression\nOverall Accuracy: {lr_acc * 100:.2f}%')
plt.savefig("../data/outputs/Logistic_Regression.png")
plt.close()

#### Random Forest

Training and fitting/ pickel dump/ accuracy score

In [5]:
rf = RandomForestClassifier(n_estimators= 31, max_depth=21)
rf.fit(X_train_scaled,y_train)
joblib.dump(rf,'../models/random_forest.pkl')

rf_pred = rf.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)
print(f"RandomForest Accuracy: {rf_acc * 100:.2f}%")
print("\nRandomForestClassification Report:\n", classification_report(y_test, rf_pred))

RandomForest Accuracy: 91.91%

RandomForestClassification Report:
               precision    recall  f1-score   support

           0       0.88      0.94      0.91      8114
           1       0.95      0.90      0.93     11070

    accuracy                           0.92     19184
   macro avg       0.92      0.92      0.92     19184
weighted avg       0.92      0.92      0.92     19184



In [6]:
# Assuming 'rf' or 'best_rf' is your trained Random Forest
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importances)

                  Feature  Importance
0              url_length    0.115312
10       suspicious_words    0.110124
32      unique_char_ratio    0.105949
14                entropy    0.096339
27          longest_token    0.094867
12            path_length    0.077833
30    max_consecutive_rep    0.077568
11                 digits    0.056734
22             path_depth    0.048868
3                     dot    0.044392
23            digit_ratio    0.040656
18          special_chars    0.027160
2                has_dash    0.026275
16             paramaters    0.025045
20  brand_domain_mismatch    0.021682
19           brand_in_url    0.020870
9         redirect_symbol    0.005088
29         hex_char_count    0.003397
5      shortening_service    0.000733
31           has_fragment    0.000650
1                  has_at    0.000292
17           has_punycode    0.000141
6   double_slash_redirect    0.000026
4                  has_ip    0.000000
13         suspicious_tld    0.000000
8           

Confusion Matrix and heatmap

In [7]:
rfcm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(6,4))
sns.heatmap(rfcm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix for RandomForest\nOverall Accuracy: {rf_acc * 100:.2f}%')
plt.savefig("../data/outputs/Random_Forest.png")
plt.close()

#### SVM
Training and fitting/ pickel dump/ accuracy score

In [8]:
svm = SVC(kernel='rbf', cache_size=2000)
svm.fit(X_train_scaled, y_train)
joblib.dump(svm,'../models/svm.pkl')

svm_pred = svm.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)
print(f"SVM Accuracy: {svm_acc * 100:.2f}%")
print("\nSVM Classification Report:\n", classification_report(y_test, svm_pred))

SVM Accuracy: 86.87%

SVM Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.90      0.85      8114
           1       0.92      0.84      0.88     11070

    accuracy                           0.87     19184
   macro avg       0.87      0.87      0.87     19184
weighted avg       0.87      0.87      0.87     19184



Confusion Matrix and heatmap

In [9]:
svmcm = confusion_matrix(y_test, svm_pred)
plt.figure(figsize=(6,4))
sns.heatmap(svmcm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix for SVM\nOverall Accuracy: {svm_acc * 100:.2f}%')
plt.savefig("../data/outputs/SVM.png")
plt.close()